# Import 

In [8]:
import os

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from kineticanalysis.analysis.combination import combine_tracks_df_ensemble

from kineticanalysis.analysis.analysis_track import (single_track_analysis,
                                       check_track_validity)

# Input value

In [2]:
path = ""
f_name = "data.csv"

# Protein length in aa
protein_length = 490
# Suntag length in aa
suntag_length = 790
# Number of Suntag
suntag_nb = 32

# Number of missing point allowed
nb_missing_point = 5
# Force analysis even if too many missing point 
force_analysis = False

# Correct for queuing
correct_q = False
# Footprint in aa, -1 if no footprint needed
footprint = 1
# Mean ribosome occupancy
m_rib_occ = 10

# dt(sec)
dt = 0.1

# select the equation betwwen "exact", "approx", "epitope"
method = "approx"


# Analysis tracks

In [3]:
df = pd.read_csv(os.path.join(path, f_name))

In [9]:
ids_track = np.unique(df["TRACK_ID"])
first_time = True
# Analyse all tracks and save it
for i in ids_track:
    print(i)

    k = np.nan
    c = np.nan
    elongation_r = np.nan
    translation_init_r = np.nan
    perr0 = np.nan
    perr1 = np.nan
    comment = ""

    datas2 = df[(df["TRACK_ID"] == i)]

    (valid, x, y, x_fix, y_fix) = check_track_validity(datas2,
                                                       i,
                                                       normalise_intensity=1,
                                                       delta_t=dt,
                                                       rtol=1e-1,
                                                       nb_missing_point=nb_missing_point,
                                                       )
    length = len(x_fix)

    if valid or force_analysis:
        if force_analysis:
            comment = "analysis forced"
        (x_auto,
         y_auto,
         k, c,
         elongation_r,
         translation_init_r,
         [perr0, perr1]) = single_track_analysis(x_fix,
                                                 y_fix,
                                                 delta_t=dt,
                                                 protein_size=protein_length,
                                                 suntag_size=suntag_length,
                                                 repetition_suntag=suntag_nb,
                                                 mm=None,
                                                 normalise_auto=True,
                                                 method=method,
                                                 simulation=False,
                                                 )

        print(k, c, elongation_r, translation_init_r)

    # Populate the dataframe
    if first_time:
        results = pd.DataFrame({
                                "id": i,
                                "dt": dt,
                                "length": length,
                                "k": k,
                                "c": c,
                                "elongation_r": elongation_r,
                                "init_translation_r": translation_init_r,
                                "perr0": perr0,
                                "perr1": perr1,
                                "comment": comment},
                               index=[0])
        first_time = False

    else:
        results = pd.concat([results,
                             pd.DataFrame(
                                 {
                                    "id": i,
                                    "dt": dt,
                                    "length": length,
                                    "k": k,
                                    "c": c,
                                    "elongation_r": elongation_r,
                                    "init_translation_r": translation_init_r,
                                    "perr0": perr0,
                                    "perr1": perr1,
                                    "comment": comment},
                                 index=[0])
                             ], ignore_index=True)


6
Gap is too big - not fix
14
20 32
2.3709546322178596 10.639861175365912 202.45009899282388 10.639861175365912
23
20 32
0.5849960123237044 309.28060913275476 820.5184136099626 309.28060913275476
80
20 32
0.1545946347400259 170.79736425853625 3104.8942986099873 170.79736425853625
90
Gap is too big - not fix
101
Gap is too big - not fix
109
Gap is too big - not fix
140
Gap is too big - not fix
143
20 32
0.10000471306599612 374.8987138599754 4799.77378349392 374.8987138599754
167
20 32
0.8366283412196366 94.72450732810573 573.7314603761284 94.72450732810573
169
20 32
1.5949348889637671 23.01604595296275 300.95272435344185 23.01604595296275
172
Gap is too big - not fix


In [12]:
results

,id,dt,length,k,c,elongation_r,init_translation_r,perr0,perr1,comment
0,6,0.1,41,NaN,NaN,NaN,NaN,NaN,NaN,
1,14,0.1,86,2.370955,10.639861,202.450099,10.639861,0.529220,2.346839,
2,23,0.1,25,0.584996,309.280609,820.518414,309.280609,0.322119,159.415091,
3,80,0.1,20,0.154595,170.797364,3104.894299,170.797364,0.074154,80.513812,
4,90,0.1,25,NaN,NaN,NaN,NaN,NaN,NaN,
5,101,0.1,19,NaN,NaN,NaN,NaN,NaN,NaN,
6,109,0.1,44,NaN,NaN,NaN,NaN,NaN,NaN,
7,140,0.1,36,NaN,NaN,NaN,NaN,NaN,NaN,
8,143,0.1,22,0.100005,374.898714,4799.773783,374.898714,0.020055,106.316806,
9,167,0.1,25,0.836628,94.724507,573.731460,94.724507,0.276529,31.962026,


In [ ]:
results.to_csv(os.path.join(path, "analysis.csv"))